<a href="https://colab.research.google.com/github/keyvanmahmoudi/Machine_Learning_Course/blob/main/solutions/Session02_preprocessing_and_pipelines_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۲: پیش‌پردازش داده و Pipeline

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session02_preprocessing_and_pipelines.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

داده و Pipeline جلسه را دوباره می‌سازیم. یک تابع کمکی هم می‌نویسیم که با گرفتن تنظیمات مختلف، Pipeline را بسازد و دقت آزمون را برگرداند:

</div>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

DATA_URL = "https://raw.githubusercontent.com/keyvanmahmoudi/Machine_Learning_Course/main/data/"
titanic = pd.read_csv(DATA_URL + "titanic.csv")
titanic["family_size"] = titanic["sibsp"] + titanic["parch"] + 1
titanic["is_alone"] = (titanic["family_size"] == 1).astype(int)

def evaluate(features, numeric, categorical, model=None, impute="median", indicator=False, scale=True):
    X_tr, X_te, y_tr, y_te = train_test_split(titanic[features], titanic["survived"], test_size=0.2,
                                              stratify=titanic["survived"], random_state=42)
    num_steps = [("impute", SimpleImputer(strategy=impute, add_indicator=indicator))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    skewed_steps = [("impute", SimpleImputer(strategy="median")),
                    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one"))]
    if scale:
        skewed_steps.append(("scale", StandardScaler()))
    prep = ColumnTransformer([
        ("num", Pipeline(num_steps), numeric),
        ("skewed", Pipeline(skewed_steps), ["fare"]),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
    ])
    pipe = Pipeline([("prep", prep), ("model", model or LogisticRegression(max_iter=1000))])
    return pipe.fit(X_tr, y_tr).score(X_te, y_te)

base_features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "family_size"]
base_numeric = ["age", "sibsp", "parch", "family_size", "pclass"]
base_categorical = ["sex", "embarked"]
print(f"Session model accuracy: {evaluate(base_features, base_numeric, base_categorical):.3f}")

Session model accuracy: 0.810


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

در `numeric_pipeline` به‌جای میانه از میانگین استفاده کنید و `add_indicator=True` را به `SimpleImputer` اضافه کنید. دقت تغییر می‌کند؟

</div>

In [2]:
for impute, indicator in [("median", False), ("mean", False), ("mean", True)]:
    acc = evaluate(base_features, base_numeric, base_categorical, impute=impute, indicator=indicator)
    print(f"strategy={impute:<6} add_indicator={indicator!s:<5}  accuracy={acc:.3f}")

strategy=median add_indicator=False  accuracy=0.810
strategy=mean   add_indicator=False  accuracy=0.810
strategy=mean   add_indicator=True   accuracy=0.816


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

تفاوت‌ها بسیار کوچک‌اند. در این داده، روش پر کردن سن تأثیر زیادی ندارد. ستون نشانگر (`add_indicator`) وقتی مفید است که **گمشده بودن** خودش با هدف ارتباط داشته باشد؛ مثلاً اگر سن مسافران درجهٔ سه بیشتر ثبت نشده باشد. برای تصمیم درست، این گزینه‌ها را باید با اعتبارسنجی متقابل مقایسه کرد، نه با یک بار تقسیم.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

ویژگی جدید `is_alone` (یک اگر `family_size` برابر ۱ باشد) بسازید و به مدل اضافه کنید.

</div>

In [3]:
acc = evaluate(base_features + ["is_alone"], base_numeric + ["is_alone"], base_categorical)
print(f"With is_alone: {acc:.3f}")
print(titanic.groupby("is_alone")["survived"].mean().round(3))

With is_alone: 0.821
is_alone
0    0.506
1    0.304
Name: survived, dtype: float64


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

مسافران تنها نرخ نجات کمتری داشته‌اند، ولی این اطلاعات تا حد زیادی در `family_size` وجود دارد؛ برای همین افزودن `is_alone` دقت را چندان تغییر نمی‌دهد. مهندسی ویژگی وقتی بیشترین اثر را دارد که اطلاعات **جدیدی** بسازد که مدل خودش نمی‌تواند به‌راحتی از ویژگی‌های موجود به دست آورد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

ستون `alive` را عمداً به ویژگی‌ها اضافه کنید و دقت را ببینید. چرا این عدد بی‌ارزش است؟

</div>

In [4]:
acc = evaluate(base_features + ["alive"], base_numeric, base_categorical + ["alive"])
print(f"With the leaked 'alive' column: {acc:.3f}")
print(pd.crosstab(titanic["alive"], titanic["survived"]))

With the leaked 'alive' column: 1.000
survived    0    1
alive             
no        549    0
yes         0  342


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

دقت به ۱۰۰٪ می‌رسد، چون `alive` همان هدف است با نام دیگر (جدول بالا نشان می‌دهد «yes» دقیقاً همان ۱ است). در لحظهٔ پیش‌بینی برای یک مسافر جدید، این اطلاعات را نداریم؛ پس مدل در عمل کاملاً بی‌فایده است. این **نشت هدف** است و هر نتیجهٔ «بیش از حد خوب» باید ما را به وجود آن مشکوک کند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

`LogisticRegression` را با `KNeighborsClassifier` عوض کنید. سپس `StandardScaler` را از Pipeline حذف کنید. چه اتفاقی برای دقت k-NN می‌افتد و چرا؟

</div>

In [5]:
from sklearn.model_selection import cross_val_score

knn = KNeighborsClassifier(n_neighbors=5)
print(f"k-NN with scaling:    {evaluate(base_features, base_numeric, base_categorical, model=knn):.3f}")
print(f"k-NN without scaling: {evaluate(base_features, base_numeric, base_categorical, model=knn, scale=False):.3f}")

k-NN with scaling:    0.782
k-NN without scaling: 0.782


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

در Pipeline جلسه، حذف `StandardScaler` اینجا تفاوتی ایجاد نکرد. چرا؟ چون `fare` با لگاریتم فشرده شده و بازهٔ بقیهٔ ویژگی‌ها (به‌جز سن) کوچک است. ولی این نتیجه روی **یک** تقسیم است. همین مقایسه را با `fare` خام و با اعتبارسنجی متقابل ۵ بخشی تکرار می‌کنیم:

</div>

In [6]:
X_all, y_all = titanic[base_features], titanic["survived"]
for scale in [True, False]:
    num_steps = [("impute", SimpleImputer(strategy="median"))] + ([("scale", StandardScaler())] if scale else [])
    prep = ColumnTransformer([
        ("num", Pipeline(num_steps), base_numeric + ["fare"]),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), base_categorical),
    ])
    scores = cross_val_score(Pipeline([("prep", prep), ("model", knn)]), X_all, y_all, cv=5)
    print(f"raw fare, scaling={scale!s:<5}  5-fold CV accuracy = {scores.mean():.3f}")

raw fare, scaling=True   5-fold CV accuracy = 0.804


raw fare, scaling=False  5-fold CV accuracy = 0.708


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با `fare` خام، حذف مقیاس‌بندی دقت k-NN را حدود ۱۰ درصد پایین می‌آورد. k-NN بر پایهٔ **فاصله** کار می‌کند؛ بدون مقیاس‌بندی، `fare` (تا ۵۰۰) و `age` (تا ۸۰) بر فاصله غلبه می‌کنند و ویژگی مهمی مثل جنسیت (صفر و یک بعد از One-Hot) تقریباً نادیده گرفته می‌شود. رگرسیون لجستیک به این اندازه حساس نیست، چون برای هر ویژگی ضریب جداگانه‌ای یاد می‌گیرد. درس دوم: یک تقسیم ممکن است اثر واقعی را پنهان کند؛ مقایسه‌ها را با اعتبارسنجی متقابل انجام دهید.

</div>